Imports


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output
import os
%matplotlib inline

## Bresenham algorithm

Returns grid points lying on a line between two coordinates.

**Args:**
- `x0, y0` – start point
- `x1, y1` – end point

**Returns:**
- list of `(x, y)` points forming the discrete line

In [2]:
def bresenham(x0, y0, x1, y1):
    points = []

    delta_x = abs(x1 - x0)
    delta_y = abs(y1 - y0)

    step_x = 1 if x0 < x1 else -1
    step_y = 1 if y0 < y1 else -1

    # initial error (difference between x and y)
    balance = delta_x - delta_y

    while True:
        points.append((x0, y0))

        if x0 == x1 and y0 == y1:
            break

        doubled_balance = 2 * balance

        # horizontal move
        if doubled_balance > -delta_y:
            balance -= delta_y
            x0 += step_x

        # vertical move
        if doubled_balance < delta_x:
            balance += delta_x
            y0 += step_y

    return points


## Generating sinogram

Builds a sinogram by simulating projections through the image.

**Args:**
- `bitmap` – 2D grayscale image array `[H x W]`
- `angles` – array of projection angles (degrees)
- `n_detectors` – number of detectors
- `arc_span_deg` – emiters/detectors span arc (degrees)

**Returns:**
- `sinogram` – 2D array `[angles x detectors]`, normalized to `[0,1]`

In [3]:
def generate_sinogram(bitmap, angles, n_detectors, arc_span_deg):

    H, W = bitmap.shape
    cx, cy = (W - 1) / 2.0, (H - 1) / 2.0
    R = min(W, H) / 2.0
    arc_span_rad = np.radians(arc_span_deg) # full span of the parallel beam array
    max_ray_offset = R * np.sin(arc_span_rad / 2) # max lateral offset from center ray

    sinogram = np.zeros((len(angles), n_detectors))

    for i, alpha_deg in enumerate(angles):
        alpha = np.radians(alpha_deg) # current angle in radians

        for j in range(n_detectors):
            # detectors are evenly spaced along distance s
            if n_detectors > 1:
                ray_lateral_offset = -max_ray_offset + j * 2 * max_ray_offset / (n_detectors - 1)
            else:
                ray_lateral_offset = 0.0

            # convert lateral offset to angular offset on the circle
            arc_angle_offset = np.arcsin(np.clip(ray_lateral_offset / R, -1.0, 1.0))

            # position of emitter on the circle
            src_x = cx + R * np.cos(alpha + arc_angle_offset)
            src_y = cy + R * np.sin(alpha + arc_angle_offset)
            # position of detector on the opposite side
            det_x = cx + R * np.cos(alpha + np.pi - arc_angle_offset)
            det_y = cy + R * np.sin(alpha + np.pi - arc_angle_offset)

            # find pixels on the line between emitter and detector
            ray_pixels = bresenham(int(round(src_x)), int(round(src_y)),
                               int(round(det_x)), int(round(det_y)))

            # sum pixel values along this ray
            sinogram[i, j] = sum(
                bitmap[py, px]
                for px, py in ray_pixels
                if 0 <= px < W and 0 <= py < H
            )

    # normalization
    if sinogram.max() > 0:
        sinogram /= sinogram.max()
    return sinogram


## Ramp filter (Ram-Lak)

Applies a ramp filter to each projection in the frequency domain.

**Args:**
- `sinogram` – 2D array `[n_angles x n_detectors]`

**Returns:**
- `filtered` – filtered sinogram of the same shape

In [4]:
def filter_sinogram(sinogram):
    n_detector_positions = sinogram.shape[1] # number of detectors (points in one projection)
    frequencies = np.fft.fftfreq(n_detector_positions) # frequency values for FFT
    ramp_filter = np.abs(frequencies) # ramp filter (Ram-Lak)

    filtered = np.zeros_like(sinogram)

    for i in range(sinogram.shape[0]):
        projection_fft = np.fft.fft(sinogram[i]) # convert projection to frequency domain
        projection_fft_filtered = projection_fft * ramp_filter # apply filter (multiply by ramp)
        filtered[i] = np.real(np.fft.ifft(projection_fft_filtered)) # go back to normal domain

    return filtered


## Backprojection

Reconstructs an image from a sinogram.

**Args:**
- `sinogram` – 2D array `[n_angles x n_detectors]`
- `angles` – projection angles (degrees)
- `n_detectors` – number of detectors
- `arc_span_deg` – emiters/detectors span arc(degrees)
- `H, W` – output image dimensions

**Returns:**
- `reconstruction` – 2D image `[H x W]`, normalized to `[0,1]`Reconstruction



In [5]:
def backproject(sinogram, angles, n_detectors, arc_span_deg, H, W):
    reconstruction = np.zeros((H, W))
    cx, cy = (W - 1) / 2.0, (H - 1) / 2.0
    R = min(W, H) / 2.0
    arc_span_rad = np.radians(arc_span_deg)
    max_ray_offset = R * np.sin(arc_span_rad / 2) # same range as in sinogram

    # create grid with pixel coordinates
    pixel_xs = np.arange(W, dtype=np.float64)
    pixel_ys = np.arange(H, dtype=np.float64)
    grid_x, grid_y = np.meshgrid(pixel_xs, pixel_ys)

    # lateral positions of all detectors
    detector_offsets = np.linspace(-max_ray_offset, max_ray_offset, n_detectors) if n_detectors > 1 else np.array([0.0])

    for i, alpha_deg in enumerate(angles):
        alpha = np.radians(alpha_deg)

        # direction perpendicular to current projection (lateral axis)
        lateral_dir_x = -np.sin(alpha)
        lateral_dir_y =  np.cos(alpha)

        # compute lateral distance from center ray for every pixel
        pixel_lateral_dist = (grid_x - cx) * lateral_dir_x + (grid_y - cy) * lateral_dir_y

        # take values from sinogram (with interpolation)
        ray_contributions = np.interp(pixel_lateral_dist.ravel(), detector_offsets, sinogram[i], left=0.0, right=0.0)
        # add contribution to reconstruction
        reconstruction += ray_contributions.reshape(H, W)

    # clip negative values
    reconstruction = np.maximum(reconstruction, 0)
    mx = reconstruction.max()
    if mx > 0:
        reconstruction /= mx
    return reconstruction


## Mean Squared Error (MSE)

Computes average squared difference between two images.

**Args:**
- `img_ref` – reference image
- `img_out` – reconstructed image

**Returns:**
- `MSE_value`

In [6]:
def mse(img_ref, img_out):
    reference = np.clip(img_ref, 0, 1)
    reconstructed = np.clip(img_out, 0, 1)
    # resize output if padding changed dimensions
    if reference.shape != reconstructed.shape:
        reconstructed_pil = Image.fromarray((reconstructed * 255).astype(np.uint8))
        reconstructed = np.array(reconstructed_pil.resize((reference.shape[1], reference.shape[0]),
                           Image.BILINEAR)) / 255.0
    return float(np.mean((reference - reconstructed) ** 2))


## Image loading and preprocessing

Loads image, converts to grayscale, normalizes and pads to square.

In [7]:
def get_image_list(folder='img'):
    """Return all .jpg filenames found in the given folder."""
    supported_extensions = ('.jpg')
    if not os.path.exists(folder):
        return []
    return [f for f in os.listdir(folder) if f.lower().endswith(supported_extensions)]

image_options = get_image_list()


def prepare_bitmap(img_path):
    """
    Load an image as a square grayscale float array in [0, 1].

    The image is padded to a square whose side equals the diagonal of the
    original (times 1.1), so that the scanner circle fits inside
    the image regardless of rotation angle.
    """
    pil_image = Image.open(img_path).convert("L")
    grayscale_array = np.array(pil_image) / 255.0
    h, w = grayscale_array.shape

    # square canvas large enough to contain all rotations without clipping
    canvas_size = int(np.ceil(np.sqrt(h**2 + w**2) * 1.1))

    canvas = np.zeros((canvas_size, canvas_size))

    # centre the original image on the canvas
    y_offset = (canvas_size - h) // 2
    x_offset = (canvas_size - w) // 2
    canvas[y_offset:y_offset+h, x_offset:x_offset+w] = grayscale_array

    return canvas


bitmap = prepare_bitmap("img/Shepp_logan.jpg")
H, W = bitmap.shape


## User interface

Provides interactive controls for reconstruction.


In [8]:
style  = {'description_width': '140px'}
layout = widgets.Layout(width='500px')

dropdown_img = widgets.Dropdown(
    options=image_options,
    value=image_options[0] if image_options else None,
    description='Select Image:',
    style=style, layout=layout
)


def on_image_change(change):
    """Reload bitmap when a different image is selected from the dropdown."""
    global bitmap, H, W
    if change['new']:
        img_path = os.path.join('img', change['new'])
        bitmap = prepare_bitmap(img_path)
        H, W = bitmap.shape
        state.update(sinogram=None, angles=None)
        status_lbl.value = f'Loaded: {change["new"]}. Ready to generate.'
        with out:
            clear_output()
            fig, ax = plt.subplots(figsize=(4,4))
            ax.imshow(bitmap, cmap='gray')
            ax.set_title("New image loaded")
            ax.axis('off')
            plt.show()


slider_a_delta = widgets.FloatSlider(value=1.0, min=0.5, max=5.0, step=0.5,
    description='delta_a [deg]', continuous_update=False, style=style, layout=layout)
slider_n_detectors = widgets.IntSlider(value=180, min=90, max=360, step=10,
    description='n detectors', continuous_update=False, style=style, layout=layout)
slider_l_degrees = widgets.IntSlider(value=180, min=45, max=270, step=5,
    description='l [deg]', continuous_update=False, style=style, layout=layout)
slider_level  = widgets.IntSlider(value=1, min=1, max=180, step=1,
    description='progress:', continuous_update=False,
    style={'description_width': '120px'}, layout=widgets.Layout(width='500px'),
    disabled=True)

chk_filter  = widgets.Checkbox(value=False, description='Use filter',
    layout=widgets.Layout(width='200px'))

btn_gen = widgets.Button(description='Generate', button_style='primary',
    layout=widgets.Layout(width='120px'))
btn_full = widgets.Button(description='Full image', button_style='success',
    layout=widgets.Layout(width='130px'), disabled=True)
progress = widgets.IntProgress(value=0, min=0, max=100, description='Progress:',
    bar_style='info', layout=widgets.Layout(width='500px'))
status_lbl = widgets.Label(value='Choose parameters and generate image')

out = widgets.Output()

# shared state between UI callbacks
state = dict(sinogram=None, angles=None, n_det=None, l=None)


def draw(sinogram, reconstruction, n_angles_shown, total_angles):
    """Display input image (with scanner overlay), sinogram, and reconstruction side by side."""
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))

    axes[0].imshow(bitmap, cmap='gray', vmin=0, vmax=1)

    if n_angles_shown > 0:
        alpha_deg = state['angles'][n_angles_shown-1]
        arc_span_deg = state['l']
        cx, cy = (W - 1) / 2.0, (H - 1) / 2.0
        scanner_radius = min(W, H) / 2.0
        arc_span_rad = np.radians(arc_span_deg)
        alpha = np.radians(alpha_deg)

        # Scanner circle outline
        circle_angles = np.linspace(0, 2 * np.pi, 300)
        axes[0].plot(cx + scanner_radius * np.cos(circle_angles),
                     cy + scanner_radius * np.sin(circle_angles),
                     'gray', linewidth=0.8, alpha=0.4)

        # Angular offsets for visualisation
        if state['n_det'] > 1:
            arc_angle_offsets = np.linspace(-arc_span_rad/2, arc_span_rad/2, state['n_det'])
        else:
            arc_angle_offsets = np.array([0.0])

        src_xs = cx + scanner_radius * np.cos(alpha + arc_angle_offsets)
        src_ys = cy + scanner_radius * np.sin(alpha + arc_angle_offsets)
        det_xs = cx + scanner_radius * np.cos(alpha + np.pi - arc_angle_offsets)
        det_ys = cy + scanner_radius * np.sin(alpha + np.pi - arc_angle_offsets)

        # Draw a subset of rays for clarity
        ray_display_step = max(1, state['n_det'] // 12)
        for j in range(0, state['n_det'], ray_display_step):
            axes[0].plot([src_xs[j], det_xs[j]], [src_ys[j], det_ys[j]],
                         'y-', alpha=0.25, linewidth=0.7)

        # Emitter arc
        emitter_arc_angles = np.linspace(alpha - arc_span_rad/2, alpha + arc_span_rad/2, 100)
        axes[0].plot(cx + scanner_radius * np.cos(emitter_arc_angles),
                     cy + scanner_radius * np.sin(emitter_arc_angles),
                     'b-', linewidth=3, label='Emitters')

        # Detector arc
        detector_arc_angles = np.linspace(alpha + np.pi - arc_span_rad/2, alpha + np.pi + arc_span_rad/2, 100)
        axes[0].plot(cx + scanner_radius * np.cos(detector_arc_angles),
                     cy + scanner_radius * np.sin(detector_arc_angles),
                     'r-', linewidth=3, label='Detectors')

        # Centre marker of detector arc
        detector_center_angle = alpha + np.pi
        detector_center_x = cx + scanner_radius * np.cos(detector_center_angle)
        detector_center_y = cy + scanner_radius * np.sin(detector_center_angle)
        axes[0].plot(detector_center_x, detector_center_y, 'r^', markersize=10, zorder=5)

        # Centre marker of emitter arc
        emitter_center_x = cx + scanner_radius * np.cos(alpha)
        emitter_center_y = cy + scanner_radius * np.sin(alpha)
        axes[0].plot(emitter_center_x, emitter_center_y, 'b^', markersize=10, zorder=5)

        axes[0].legend(loc='upper right', fontsize=7)

    axes[0].set_title('Input image')
    axes[0].axis('off')

    axes[1].imshow(sinogram, cmap='gray', aspect='auto')
    axes[1].set_title(f'Sinogram ({n_angles_shown}/{total_angles})')

    axes[2].imshow(reconstruction, cmap='gray', vmin=0, vmax=1)
    axes[2].set_title(f'Reconstruction ({n_angles_shown}/{total_angles})')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()


def on_gen(b):
    """Generate sinogram and full reconstruction when the Generate button is clicked."""
    state.update(sinogram=None, angles=None)

    with out:
        clear_output(wait=True)

    projection_angles = np.arange(0, 180, float(slider_a_delta.value))
    n_detectors = int(slider_n_detectors.value)
    arc_span_deg = float(slider_l_degrees.value)
    total_angles = len(projection_angles)

    btn_gen.disabled = True
    slider_level.disabled = True
    btn_full.disabled = True
    progress.max = total_angles
    progress.value = 0
    status_lbl.value = 'Generating sinogram...'

    sinogram = generate_sinogram(bitmap, projection_angles, n_detectors, arc_span_deg)
    if chk_filter.value:
        sinogram_for_bp = filter_sinogram(sinogram)
    else:
        sinogram_for_bp = sinogram

    progress.value = total_angles // 2
    status_lbl.value = 'Backprojecting...'
    reconstruction = backproject(sinogram_for_bp, projection_angles, n_detectors, arc_span_deg, H, W)
    progress.value = total_angles

    state.update(sinogram=sinogram_for_bp, angles=projection_angles, n_det=n_detectors, l=arc_span_deg)
    slider_level.max = total_angles
    slider_level.value = total_angles
    slider_level.disabled = False
    btn_full.disabled = False
    btn_gen.disabled = False
    status_lbl.value = f'Done  {total_angles} angles | {n_detectors} det | l={arc_span_deg:.0f} deg'

    with out:
        clear_output(wait=True)
        draw(sinogram_for_bp, reconstruction, total_angles, total_angles)


def on_prog(change):
    """Re-run backprojection with only the first n angles when the progress slider moves."""
    if state['sinogram'] is None:
        return
    n_angles = min(change['new'], len(state['angles']))
    sinogram_partial = state['sinogram'][:n_angles]
    reconstruction_partial = backproject(sinogram_partial, state['angles'][:n_angles], state['n_det'], state['l'], H, W)
    with out:
        clear_output(wait=True)
        draw(sinogram_partial, reconstruction_partial, n_angles, len(state['angles']))


def on_full(b):
    """Jump the progress slider to the final angle (show full reconstruction)."""
    if state['angles'] is not None:
        slider_level.value = len(state['angles'])


# clear previous callbacks to avoid duplicates
btn_gen._click_handlers.callbacks.clear()
btn_full._click_handlers.callbacks.clear()
slider_level.unobserve_all()
dropdown_img.unobserve_all()

btn_gen.on_click(on_gen)
slider_level.observe(on_prog, names='value')
btn_full.on_click(on_full)
dropdown_img.observe(on_image_change, names='value')

display(widgets.VBox([
    widgets.HTML('<b>Input Data</b>'),
    dropdown_img,
    widgets.HTML('<b>Parameters</b>'),
    slider_a_delta, slider_n_detectors, slider_l_degrees,
    chk_filter,
    widgets.HBox([btn_gen, widgets.Label('   '), status_lbl]),
    progress,
    widgets.HBox([slider_level, btn_full]),
    out,
]))


## Statistical analysis

Evaluates reconstruction quality using MSE.


Analyzes the effect of:
- number of projection angles
- number of detectors
- fan angle
- filtering

In [9]:
def run_statistical_analysis():
    """Run four quality analyses and plot results:
    1. MSE vs number of projection angles used in reconstruction
    2. MSE vs detector count
    3. MSE vs fan angle l_degrees
    4. Filter impact (MSE with / without ramp filter)
    """
    if state['sinogram'] is None:
        print("Generate a sinogram first.")
        return

    print("Running analysis...")
    projection_angles = state['angles']
    n_detectors = state['n_det']
    arc_span_deg = state['l']
    total_angles = len(projection_angles)

    # 1. MSE vs number of angles used in reconstruction
    mse_per_angle_count = []
    angle_counts = np.linspace(1, total_angles, 10, dtype=int)
    for n_angles in angle_counts:
        reconstruction = backproject(state['sinogram'][:n_angles], projection_angles[:n_angles], n_detectors, arc_span_deg, H, W)
        mse_per_angle_count.append(mse(bitmap, reconstruction))

    # 2. MSE vs detector count
    detector_counts = [90, 180, 270, 360]
    mse_per_detector_count = []
    for det_count in detector_counts:
        sinogram = generate_sinogram(bitmap, projection_angles, det_count, arc_span_deg)
        if chk_filter.value: sinogram = filter_sinogram(sinogram)
        reconstruction = backproject(sinogram, projection_angles, det_count, arc_span_deg, H, W)
        mse_per_detector_count.append(mse(bitmap, reconstruction))

    # 3. MSE vs arc span
    arc_span_values = [45, 90, 180, 270]
    mse_per_arc_span = []
    for arc_span in arc_span_values:
        sinogram = generate_sinogram(bitmap, projection_angles, n_detectors, arc_span)
        if chk_filter.value: sinogram = filter_sinogram(sinogram)
        reconstruction = backproject(sinogram, projection_angles, n_detectors, arc_span, H, W)
        mse_per_arc_span.append(mse(bitmap, reconstruction))

    # 4. Ramp filter impact
    sinogram_unfiltered = generate_sinogram(bitmap, projection_angles, n_detectors, arc_span_deg)
    mse_without_filter = mse(bitmap, backproject(sinogram_unfiltered, projection_angles, n_detectors, arc_span_deg, H, W))
    mse_with_filter = mse(bitmap, backproject(filter_sinogram(sinogram_unfiltered), projection_angles, n_detectors, arc_span_deg, H, W))

    fig, axes = plt.subplots(2, 2, figsize=(18, 10))

    axes[0,0].plot(angle_counts, mse_per_angle_count, 'o-b')
    axes[0,0].set_title('MSE vs Number of Angles')
    axes[0,0].grid(True)

    axes[0,1].plot(detector_counts, mse_per_detector_count, 'o-g')
    axes[0,1].set_title('MSE vs Detector Count')
    axes[0,1].grid(True)

    axes[1,0].plot(arc_span_values, mse_per_arc_span, 'o-g')
    axes[1,0].set_title('MSE vs emiters/detectors arc span')
    axes[1,0].grid(True)

    axes[1,1].bar(['Off', 'On'], [mse_without_filter, mse_with_filter], color=['orange', 'skyblue'])
    axes[1,1].set_title('Filter Impact on MSE')

    plt.tight_layout()
    plt.show()


btn_analyze = widgets.Button(description='Run Analysis', button_style='info')
out_analyze = widgets.Output()


def on_analyze_clicked(b):
    with out_analyze:
        clear_output(wait=True)
        run_statistical_analysis()


btn_analyze.on_click(on_analyze_clicked)
display(btn_analyze, out_analyze)


Button(button_style='info', description='Run Analysis', style=ButtonStyle())

Output()

## DICOM handling

Saves reconstructed image as a DICOM file with basic metadata.

Allows loading and displaying DICOM images along with patient data.

In [10]:
import pydicom
from pydicom.dataset import Dataset, FileDataset, FileMetaDataset
from pydicom.uid import generate_uid, ExplicitVRLittleEndian
import datetime

In [11]:
style_d  = {'description_width': '160px'}
layout_d = widgets.Layout(width='480px')

txt_name    = widgets.Text(value='Jan Kowalski',   description='Full name',              style=style_d, layout=layout_d)
txt_birth   = widgets.Text(value='YYYYMMDD',       description='Birth date (YYYYMMDD)',  style=style_d, layout=layout_d)
txt_date    = widgets.Text(value=datetime.date.today().strftime('%Y%m%d'),
                               description='Study date (YYYYMMDD)', style=style_d, layout=layout_d)
txt_comment = widgets.Textarea(value='test comment', description='Comment',             style=style_d,
                               layout=widgets.Layout(width='480px', height='70px'))
txt_outpath = widgets.Text(value='ct_result.dcm',   description='Filename .dcm',         style=style_d, layout=layout_d)

btn_save = widgets.Button(description='Save DICOM',  button_style='warning',
                          layout=widgets.Layout(width='160px'))
btn_load = widgets.Button(description='Load DICOM',  button_style='info',
                          layout=widgets.Layout(width='160px'))
txt_loadpath = widgets.Text(value='ct_result.dcm', description='File to load',
                            style=style_d, layout=layout_d)

dicom_status = widgets.Label(value='')
out_dicom    = widgets.Output()

display(widgets.VBox([
    widgets.HTML('<b>Patient & Study Information</b>'),
    txt_name, txt_birth, txt_date, txt_comment,
    widgets.HTML('<b>Save</b>'),
    txt_outpath, btn_save,
    widgets.HTML('<b>Load</b>'),
    txt_loadpath, btn_load,
    dicom_status,
    out_dicom,
]))


In [12]:
def arr_to_uint16(arr):
    """Scale float image [0,1] to 12-bit uint16 range [0, 4095]."""
    clipped = np.clip(arr, 0, 1)
    return (clipped * 4095).astype(np.uint16)


def build_dicom(pixel_uint16, patient_name, birth_date, study_date, comment):
    """Construct a minimal CT DICOM dataset from pixel data and patient metadata."""
    rows, cols = pixel_uint16.shape
    sop_instance_uid = generate_uid()

    file_meta = FileMetaDataset()
    file_meta.FileMetaInformationGroupLength = 0
    file_meta.FileMetaInformationVersion = b'\x00\x01'
    file_meta.MediaStorageSOPClassUID = '1.2.840.10008.5.1.4.1.1.2'  # CT Image Storage
    file_meta.MediaStorageSOPInstanceUID = sop_instance_uid
    file_meta.TransferSyntaxUID = ExplicitVRLittleEndian
    file_meta.ImplementationClassUID = '1.2.3.4.5'

    dicom_dataset = FileDataset(None, {}, file_meta=file_meta, preamble=b'\x00' * 128)

    # Patient tags
    dicom_dataset.PatientName = patient_name
    dicom_dataset.PatientID = "123456"
    dicom_dataset.PatientBirthDate = birth_date
    dicom_dataset.PatientSex = "O"

    # Study / series tags
    dicom_dataset.StudyDate = study_date
    dicom_dataset.SeriesDate = study_date
    dicom_dataset.AcquisitionDate = study_date
    dicom_dataset.ContentDate = study_date
    dicom_dataset.Modality = "CT"
    dicom_dataset.SeriesInstanceUID = generate_uid()
    dicom_dataset.StudyInstanceUID = generate_uid()
    dicom_dataset.SOPClassUID = '1.2.840.10008.5.1.4.1.1.2'
    dicom_dataset.SOPInstanceUID = sop_instance_uid
    dicom_dataset.InstanceNumber = "1"

    # Pixel format tags
    dicom_dataset.SamplesPerPixel = 1
    dicom_dataset.PhotometricInterpretation = 'MONOCHROME2'
    dicom_dataset.Rows = rows
    dicom_dataset.Columns = cols
    dicom_dataset.BitsAllocated = 16
    dicom_dataset.BitsStored = 16
    dicom_dataset.HighBit = 15
    dicom_dataset.PixelRepresentation = 0    # unsigned
    dicom_dataset.RescaleIntercept = "0"
    dicom_dataset.RescaleSlope = "1"

    dicom_dataset.PixelData = pixel_uint16.tobytes()
    return dicom_dataset


def on_save(b):
    """Save the current reconstruction as a DICOM file."""
    if state.get('sinogram') is None:
        dicom_status.value = 'Generate a reconstruction first.'
        return

    reconstruction = backproject(
        state['sinogram'], state['angles'],
        state['n_det'], state['l'], H, W
    )

    pixel_data_uint16 = arr_to_uint16(reconstruction)
    dicom_dataset = build_dicom(
        pixel_data_uint16,
        patient_name = txt_name.value,
        birth_date   = txt_birth.value,
        study_date   = txt_date.value,
        comment      = txt_comment.value,
    )

    output_path = txt_outpath.value or 'ct_result.dcm'
    pydicom.dcmwrite(output_path, dicom_dataset)
    dicom_status.value = f'Saved to {output_path}'


def on_load(b):
    """Load a DICOM file and display the image with key metadata."""
    file_path = txt_loadpath.value
    if not os.path.exists(file_path):
        dicom_status.value = f'File not found: "{file_path}"'
        return

    dicom_dataset = pydicom.dcmread(file_path)
    pixel_array = dicom_dataset.pixel_array.astype(np.float64)
    pixel_array = (pixel_array - pixel_array.min()) / (pixel_array.max() - pixel_array.min() + 1e-9)

    with out_dicom:
        clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(pixel_array, cmap='gray', vmin=0, vmax=1)
        ax.set_title(f'Loaded: {os.path.basename(file_path)}')
        ax.axis('off')
        plt.tight_layout()
        plt.show()

        # Print key DICOM metadata
        def tag(name, default='—'):
            return str(getattr(dicom_dataset, name, default))

        print(f'  Patient:     {tag("PatientName")}')
        print(f'  Birth date:  {tag("PatientBirthDate")}')
        print(f'  Study date:  {tag("StudyDate")}')
        print(f'  Comment:     {tag("ImageComments")}')

    dicom_status.value = f'Showing: {file_path}'


btn_save.on_click(on_save)
btn_load.on_click(on_load)


## Experiments

Runs experiments to study how parameters affect reconstruction quality.

Measures RMSE and visualizes results for different:
- detector counts
- projection counts
- fan angles
- filtering options

In [ ]:
btn_experiments = widgets.Button(
    description='Start experiments',
    layout=widgets.Layout(width='200px')
)

out_exp = widgets.Output()

def run_experiments(b):
    with out_exp:
        clear_output(wait=True)

        # Default parameters
        default_n_detectors = 180
        default_n_angles = 180
        default_arc_span = 180
        default_projection_angles = np.arange(0, 180, 180 / default_n_angles)

        # --- Experiment 1 ---
        print("Experiment 1: Detector count")

        detector_counts = list(range(90, 721, 90))
        rmse_per_detector_count = []
        rmse_per_detector_count_filtered = []

        n_detector_values = len(detector_counts)

        fig_nf, axes_nf = plt.subplots(2, n_detector_values, figsize=(n_detector_values * 3, 6))
        fig_f,  axes_f  = plt.subplots(2, n_detector_values, figsize=(n_detector_values * 3, 6))

        for i, det_count in enumerate(detector_counts):
            sinogram = generate_sinogram(bitmap, default_projection_angles, det_count, default_arc_span)

            reconstruction = backproject(sinogram, default_projection_angles, det_count, default_arc_span, H, W)
            rmse_value = np.sqrt(mse(bitmap, reconstruction))
            rmse_per_detector_count.append(rmse_value)

            reconstruction_filtered = backproject(filter_sinogram(sinogram), default_projection_angles, det_count, default_arc_span, H, W)
            rmse_value_filtered = np.sqrt(mse(bitmap, reconstruction_filtered))
            rmse_per_detector_count_filtered.append(rmse_value_filtered)

            axes_nf[0, i].imshow(reconstruction, cmap='gray', vmin=0, vmax=1)
            axes_nf[0, i].set_title(f'd={det_count}', fontsize=9)
            axes_nf[0, i].axis('off')
            axes_nf[1, i].text(0.5, 0.5, f'RMSE\n{rmse_value:.4f}',
                               ha='center', va='center', fontsize=10,
                               transform=axes_nf[1, i].transAxes)
            axes_nf[1, i].axis('off')

            axes_f[0, i].imshow(reconstruction_filtered, cmap='gray', vmin=0, vmax=1)
            axes_f[0, i].set_title(f'd={det_count}', fontsize=9)
            axes_f[0, i].axis('off')
            axes_f[1, i].text(0.5, 0.5, f'RMSE\n{rmse_value_filtered:.4f}',
                              ha='center', va='center', fontsize=10,
                              transform=axes_f[1, i].transAxes)
            axes_f[1, i].axis('off')

        fig_nf.suptitle('Detector count – No filter')
        fig_nf.tight_layout()
        plt.figure(fig_nf.number)
        plt.show()

        fig_f.suptitle('Detector count – Filtered')
        fig_f.tight_layout()
        plt.figure(fig_f.number)
        plt.show()

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        ax1.plot(detector_counts, rmse_per_detector_count, 'o-b')
        ax1.set_title('RMSE vs Detector Count – No filter')
        ax1.set_xlabel('Number of detectors')
        ax1.set_ylabel('RMSE')
        ax1.grid(True)
        ax2.plot(detector_counts, rmse_per_detector_count_filtered, 'o-c')
        ax2.set_title('RMSE vs Detector Count – Filtered')
        ax2.set_xlabel('Number of detectors')
        ax2.set_ylabel('RMSE')
        ax2.grid(True)
        plt.tight_layout()
        plt.show()


        # --- Experiment 2 ---
        print("Experiment 2: Number of angles")

        angle_counts = list(range(90, 721, 90))
        rmse_per_angle_count = []
        rmse_per_angle_count_filtered = []

        n_angle_values = len(angle_counts)

        fig_nf, axes_nf = plt.subplots(2, n_angle_values, figsize=(n_angle_values * 3, 6))
        fig_f,  axes_f  = plt.subplots(2, n_angle_values, figsize=(n_angle_values * 3, 6))

        for i, n_angles in enumerate(angle_counts):
            projection_angles = np.arange(0, 180, 180 / n_angles)
            sinogram = generate_sinogram(bitmap, projection_angles, default_n_detectors, default_arc_span)

            reconstruction = backproject(sinogram, projection_angles, default_n_detectors, default_arc_span, H, W)
            rmse_value = np.sqrt(mse(bitmap, reconstruction))
            rmse_per_angle_count.append(rmse_value)

            reconstruction_filtered = backproject(filter_sinogram(sinogram), projection_angles, default_n_detectors, default_arc_span, H, W)
            rmse_value_filtered = np.sqrt(mse(bitmap, reconstruction_filtered))
            rmse_per_angle_count_filtered.append(rmse_value_filtered)

            axes_nf[0, i].imshow(reconstruction, cmap='gray', vmin=0, vmax=1)
            axes_nf[0, i].set_title(f's={n_angles}', fontsize=9)
            axes_nf[0, i].axis('off')
            axes_nf[1, i].text(0.5, 0.5, f'RMSE\n{rmse_value:.4f}',
                               ha='center', va='center', fontsize=10,
                               transform=axes_nf[1, i].transAxes)
            axes_nf[1, i].axis('off')

            axes_f[0, i].imshow(reconstruction_filtered, cmap='gray', vmin=0, vmax=1)
            axes_f[0, i].set_title(f's={n_angles}', fontsize=9)
            axes_f[0, i].axis('off')
            axes_f[1, i].text(0.5, 0.5, f'RMSE\n{rmse_value_filtered:.4f}',
                              ha='center', va='center', fontsize=10,
                              transform=axes_f[1, i].transAxes)
            axes_f[1, i].axis('off')

        fig_nf.suptitle('Projection angles – No filter')
        fig_nf.tight_layout()
        plt.figure(fig_nf.number)
        plt.show()

        fig_f.suptitle('Projection angles – Filtered')
        fig_f.tight_layout()
        plt.figure(fig_f.number)
        plt.show()

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        ax1.plot(angle_counts, rmse_per_angle_count, 'o-g')
        ax1.set_title('RMSE vs Number of Angles – No filter')
        ax1.set_xlabel('Number of angles')
        ax1.set_ylabel('RMSE')
        ax1.grid(True)
        ax2.plot(angle_counts, rmse_per_angle_count_filtered, 'o-', color='lime')
        ax2.set_title('RMSE vs Number of Angles – Filtered')
        ax2.set_xlabel('Number of angles')
        ax2.set_ylabel('RMSE')
        ax2.grid(True)
        plt.tight_layout()
        plt.show()


        # --- Experiment 3 ---
        print("Experiment 3: Arc span")

        arc_span_values = list(range(45, 271, 45))
        rmse_per_arc_span = []
        rmse_per_arc_span_filtered = []

        n_arc_span_values = len(arc_span_values)

        fig_nf, axes_nf = plt.subplots(2, n_arc_span_values, figsize=(n_arc_span_values * 3, 6))
        fig_f,  axes_f  = plt.subplots(2, n_arc_span_values, figsize=(n_arc_span_values * 3, 6))

        for i, arc_span in enumerate(arc_span_values):
            sinogram = generate_sinogram(bitmap, default_projection_angles, default_n_detectors, arc_span)

            reconstruction = backproject(sinogram, default_projection_angles, default_n_detectors, arc_span, H, W)
            rmse_value = np.sqrt(mse(bitmap, reconstruction))
            rmse_per_arc_span.append(rmse_value)

            reconstruction_filtered = backproject(filter_sinogram(sinogram), default_projection_angles, default_n_detectors, arc_span, H, W)
            rmse_value_filtered = np.sqrt(mse(bitmap, reconstruction_filtered))
            rmse_per_arc_span_filtered.append(rmse_value_filtered)

            axes_nf[0, i].imshow(reconstruction, cmap='gray', vmin=0, vmax=1)
            axes_nf[0, i].set_title(f'l={arc_span}°', fontsize=9)
            axes_nf[0, i].axis('off')
            axes_nf[1, i].text(0.5, 0.5, f'RMSE\n{rmse_value:.4f}',
                               ha='center', va='center', fontsize=10,
                               transform=axes_nf[1, i].transAxes)
            axes_nf[1, i].axis('off')

            axes_f[0, i].imshow(reconstruction_filtered, cmap='gray', vmin=0, vmax=1)
            axes_f[0, i].set_title(f'l={arc_span}°', fontsize=9)
            axes_f[0, i].axis('off')
            axes_f[1, i].text(0.5, 0.5, f'RMSE\n{rmse_value_filtered:.4f}',
                              ha='center', va='center', fontsize=10,
                              transform=axes_f[1, i].transAxes)
            axes_f[1, i].axis('off')

        fig_nf.suptitle('Arc span – No filter')
        fig_nf.tight_layout()
        plt.figure(fig_nf.number)
        plt.show()

        fig_f.suptitle('Arc span – Filtered')
        fig_f.tight_layout()
        plt.figure(fig_f.number)
        plt.show()

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        ax1.plot(arc_span_values, rmse_per_arc_span, 'o-r')
        ax1.set_title('RMSE vs Arc Span – No filter')
        ax1.set_xlabel('Arc span (degrees)')
        ax1.set_ylabel('RMSE')
        ax1.grid(True)
        ax2.plot(arc_span_values, rmse_per_arc_span_filtered, 'o-', color='salmon')
        ax2.set_title('RMSE vs Arc Span – Filtered')
        ax2.set_xlabel('Arc span (degrees)')
        ax2.set_ylabel('RMSE')
        ax2.grid(True)
        plt.tight_layout()
        plt.show()


        # --- Experiment 4 ---
        print("Experiment 4: Filter comparison")

        best_params = dict(n_det=360, l=270)
        full_projection_angles = np.arange(0, 180, 180 / 360)

        for img_path in ["img/CT_ScoutView.jpg", "img/Shepp_logan.jpg"]:
            test_bitmap = prepare_bitmap(img_path)
            h, w = test_bitmap.shape

            sinogram = generate_sinogram(test_bitmap, full_projection_angles, best_params['n_det'], best_params['l'])

            reconstruction_no_filter = backproject(sinogram, full_projection_angles, best_params['n_det'], best_params['l'], h, w)
            reconstruction_filtered  = backproject(filter_sinogram(sinogram), full_projection_angles,
                                  best_params['n_det'], best_params['l'], h, w)

            print(img_path)
            print("RMSE no filter:", np.sqrt(mse(test_bitmap, reconstruction_no_filter)))
            print("RMSE filter:", np.sqrt(mse(test_bitmap, reconstruction_filtered)))

            fig, axes = plt.subplots(1, 3, figsize=(12, 4))
            axes[0].imshow(test_bitmap, cmap='gray')
            axes[0].set_title('Original')
            axes[0].axis('off')
            axes[1].imshow(reconstruction_no_filter, cmap='gray')
            axes[1].set_title('No filter')
            axes[1].axis('off')
            axes[2].imshow(reconstruction_filtered, cmap='gray')
            axes[2].set_title('Filter')
            axes[2].axis('off')
            plt.show()


btn_experiments.on_click(run_experiments)

display(btn_experiments, out_exp)

Button(description='Start experiments', layout=Layout(width='200px'), style=ButtonStyle())

Output()